In [8]:
from collections import Counter
from itertools import zip_longest, chain
from typing import Tuple
import json, math

# Add some new method to class Partition
from sage.combinat.partition import Partition

def col(self, square):
    return [i for i in self.cells() if i[1] == square[1] and i[0] != square[0]]
Partition.col = col

def row(self, square):
    return [i for i in self.cells() if i[0] == square[0] and i[1] != square[1]]
Partition.row = row

def arm(self, square):
    if square[0] >= len(self):
        res = - square[1] - 1
    else:
        res = self[square[0]] - square[1] - 1
    return res
Partition.arm = arm

def leg(self, square):
    if square[1] >= len(self.conjugate()):
        res = - square[0] - 1
    else:
        res = self.conjugate()[square[1]] - square[0] - 1
    return res
Partition.leg = leg

def x_len(self, square):
    return square[1]
Partition.x_len = x_len

def y_len(self, square):
    return square[0]
Partition.y_len = y_len

def check_containing(self, p):
    if self.size() >= sum(p) and self.length() >= len(p)\
    and all(self.components()[0][i] >= p[i] for i in range(len(p))):
        return True
    else:
        return False
Partition.check_containing = check_containing

def check_contained(self, p):
    return Partition(p).check_containing(self.components()[0])
Partition.check_contained = check_contained

def subtracting(self, p):
    if self.check_containing(p):
        '''
        Example: we have two partition mu1 = [7,4,3,2], mu2 = [6,3,1].
        diff will give the list [0,[7,6]], [1,[4,2]], [2,[3,1]], [3,[2,0]].
        it records the height of two partitions on each horizontal coordinate.
        If mu1[i] = mu2[i] the this column will be ruled out.
        
        f sends a pair with index [i,[a,b]] to boxes between [i,a] and [i,b]
        i.e. f:[i,[a,b]] --> [i,b],...,[i, a-1]
        '''
        diff = filter(lambda x:x[1][0]>x[1][1],enumerate(zip_longest(self.components()[0], p, fillvalue=0)))
        f = lambda index_pair:list(map(lambda x:[index_pair[0],x], range(index_pair[1][1],index_pair[1][0])))
        diff_box = chain(*map(f ,diff))
        return list(diff_box)
    else:
        print(f"{self.components()[0]} doesn't contain {p}.")
Partition.subtracting = subtracting

def subtracted(self, p):
    if self.check_contained(p):
        return Partition(p).subtracting(self.components()[0])
    else:
        print(f"{self.components()[0]} isn't contained in {p}.")
Partition.subtracted = subtracted

# If lambda = (1^k1, 2^k2, 3^k3, ...), this function will return k1!*k2!*k3!*...
def multiplicity_fac(self):
    multiplicities = Counter(self).values()
    return prod(map(math.factorial,multiplicities))
Partition.multiplicity_fac = multiplicity_fac

def scalar_pow(self):
    base_ring.<a> = PolynomialRing(QQ)
    p = Partition(self)
    scalar_prod = prod(self) * p.multiplicity_fac() * a^len(self)
    return scalar_prod
Partition.scalar_pow = scalar_pow

# The function H'_\lambda in wikipedia page of P normalizations - Jack polynomials
def H(self):
    base_ring.<q,t> = PolynomialRing(QQ)
    p = Partition(self)
    return prod([-(p.leg(s) + 1) * t + p.arm(s) * q for s in p.cells()])
Partition.H = H

# z_\lambda defined in Macdonald's book
def z(self):
    return prod(self)*self.multiplicity_fac()
Partition.z = z

# convert (m1,m2,...,mk) to (1^m1, 2^m2, ... , k^mk)
def multiplicity_to_partition(mul:Tuple[int]) -> type[Partition]:
    '''
    EXAMPLE:
    (0,1) -> Partition([2])
    (4,2,1) -> Partition([3,2,2,1,1,1,1])
    (1,0,2,1) -> Partition([4,3,3,1])
    '''
    res = [mul[idx]*[idx+1] for idx in range(len(mul))[::-1]]
    return Partition(chain(*res))

# convert (1^m1, 2^m2, ... , k^mk) to (m1,m2,...,mk)
def partition_to_multiplicity(part:type[Partition]) -> Tuple[int]:
    '''
    EXAMPLE:
    Partition([]) -> (0)
    Partition([2]) -> (0,1)
    Partition([3,2,2,1,1,1,1]) -> (4,2,1)
    Partition([4,3,3,1]) -> (1,0,2,1)
    '''
    if len(part) > 0:
        multiplicity = Counter(part)
        res = tuple(multiplicity.get(idx,0) for idx in range(1, part[0]+1))
    else:
        res = tuple([0])
    return res

In [9]:
# define some equivariant classes in cohomology of Hilbert schemes
class Hilb:
    def __init__(self, dim):
        self.dim = dim
        R.<q,t> = PolynomialRing(QQ);
        self.base_ring = R
        self.char = (self.base_ring.gen(), self.base_ring.gen(1))
        self.base_field = FractionField(self.base_ring);
        self.cohom = CombinatorialFreeModule(self.base_field, Partitions(self.dim))
    
    def check_fixed_point(self, mu):
        if sum(mu) == self.dim:
            return True
        else:
            print(f"{mu} is not a fixed point of Hilb_{self.dim}")
            return False
    
    # equivariant class of tangent bundle at [I_mu]
    def tangent(self, mu):
        if self.check_fixed_point(mu):
            p = Partition(mu)
            q, t = self.char
            return sum([t^p.leg(s) * q^(-p.arm(s)-1) + t^(-p.leg(s)-1) * q^p.arm(s) for s in p.cells()])
    
    # euler character of tangent bundle at [I_mu]
    def eu_tangent(self, mu):
        if self.check_fixed_point(mu):
            p = Partition(mu)
            q, t = self.char
            return prod([(p.leg(s)*t - (p.arm(s) + 1)*q) * (-(p.leg(s) + 1) * t + p.arm(s)*q) for s in p.cells()])
    
    # tautological bundle localized at [I_mu]
    def tau(self, mu):
        if self.check_fixed_point(mu):
            p = Partition(mu)
            q, t = self.char
            return sum([t^(p.x_len(s) - 1) * q^(p.y_len(s) - 1) for s in p.cells()])

# define some equivariant classes in cohomology of nested Hilbert schemes
class Nested_Hilb(Hilb):
    def __init__(self, dim, *tar_dim):
        super().__init__(dim)
        self.tar_dim = dim + 1
        
    def check_fixed_point_pair(self, mu1, mu2):
        if sum(mu1) == self.dim and sum(mu2) == self.tar_dim and Partition(mu2).check_containing(mu1):
            return True
        else:
            print(f"({mu1},{mu2}) is not fixed point \
            on the incidence Hilbert scheme Hilb_{self.dim},{self.tar_dim}")
    
    # euler class of tautological bundle on incidence hilbert scheme at (mu, lambda) where |lambda| = |mu| + 1.
    # square = lambda - mu
    def eu_tau_inc(self, mu1, mu2):
        if self.check_fixed_point(mu1) and Partition(mu2).check_containing(mu1):
            q, t = self.char
            square = Partition(mu2).subtracting(mu1)
            return square[0]*t + square[1]*q
    
    # euler class of dual of cotangent bundle at I_p1,p2
    # p1 is a partition contained in p2 by viewing Young diagram modified by only one cell.
    # p1 is fixed point on Hilb_dim
    def eu_N_dual(self, mu1, mu2):
        if self.check_fixed_point(mu1) and Partition(mu2).check_containing(mu1):
            q, t = self.char
            p1 = Partition(mu1)
            p2 = Partition(mu2)
            # chern root of N_dual at I_p1,p2
            N_dual_char = lambda s: (p1.leg(s)*t - (p2.arm(s)+1)*q) * (-(p2.leg(s)+1)*t + p1.arm(s)*q)
            return prod(list(map(N_dual_char, p1.cells())))

In [12]:
class FormalVariables():
    # scale = the number of variables you want
    def __init__(self, scale):
        self.scale = scale
    
    # this method gives list of formal variables
    def variables(self):
        polyring = PolynomialRing(QQ, self.scale, 'x')
        return [polyring.gen(index) for index in range(0, self.scale)]
    
    # convert mpoly.dict() back to a multivariable polynomial
    # we didn't check whether the number of variables is aligned with which in the dictionary
    # you need ensure this yourself
    def load_dict(self, mpoly_dict):
        mpoly = sum([value * prod([self.x(i)^key[i] for i in range(len(key))])\
                     for key, value in mpoly_dict.items()])
        return mpoly
    
    # this method returns a single variable of given index
    def x(self, index):
        return self.variables()[index]
    
    # this operation remove x_i in x_0,..., x_n, n = length -1
    def rmv(self, index, length):
        new_list = [self.x(i) for i in range(0, length) if i != index]
        return new_list + [0 for e in range(length - 1 , self.scale)]
    
    # the operation removing x_i and x_j then adding x_i + x_j within x_0,...,x_n, n = length -1
    def trans(self, index1, index2, length):
        new_list = [self.x(i) for i in range(0, length) if i != index1 and i != index2]
        new_list.append(self.x(index1) + self.x(index2))
        return new_list + [0 for e in range(length - 1 , self.scale)]
    
    # sum x_0 + ... + x_n, n = length-1
    def l1sum(self, length):
        return sum(self.variables()[:length])
    
    # square sum  x_0^2 + ... + x_n^2, n = length-1
    def sqsum(self, index):
        square_variables = [self.x(i)^2 for i in range(0, index)]
        return sum(square_variables)
    
    # return x_0,...,x_i-1,...,x_n, n = length - 1, i = index
    # index is not larger than length
    def minus1(self, index, length):
        if index < length:
            modif = [-1 if i == index else 0 for i in range(0, length)]
        else:
            raise ValueError('index should be smaller than length')
        new_list = [x+y for x,y in zip(self.variables()[:length],modif)]
        return new_list + [0 for e in range(length, self.scale)]
        

In [15]:
class f_K():
    def _init_(self):
        pass
    
    def save(self, file_path, data):
        '''
        save the result of computation of f_K into a json format file
        
        INPUT:
        - ``file_path`` -- str: path to the location you want save data
        - ``data`` -- List[Multivariable Polynomial]
        
        OUTPUT:
        - returns none
        '''
        # convert f_K into dictionary with degree as keys, coeffienct as values
        data_dict = [{str(key):str(value) for key, value in mpoly.dict().items()} for mpoly in data]
        
        with open(file_path, 'w') as file:
            json.dump(data_dict, file)
            file.close()
    
    def load(self, file_path):
        '''
        load the data of f_K in file_path into a list
        
        INPUT:
        - ``file_path`` -- str: path to the location you want save data
        
        OUTPUT:
        - returns data_reload : List[Multivariable Polynomial]
        '''
        # load the data
        with open(file_path, 'r') as file:
            data = json.load(file)
            file.close()
            
        # Process the data
        
        # step1 : extract the number of variables
        length = len(eval(list(data[0].keys())[0]))
        R = FormalVariables(length)
        
        # step2 : recover key and value into tuple and int resp.
        recovered_data = [{eval(key):eval(value) for key, value in mpoly_dict.items()} for mpoly_dict in data]
        
        # load the dict of mpoly back into multivariable polynomials
        res = [R.load_dict(mpoly_dict) for mpoly_dict in recovered_data]
        return res